# 01 — Data Engineering & Preprocessing (UBCF)

Phase 1 (user-based adaptation):
- Chunked ingestion of MovieLens 32M ratings
- ID remapping and user-focused sparse matrix
- Truncated SVD (32 latent user features)
- Content features from genres/tags for hybrid fallback

Outputs: `user_sparse.npz`, `user_latent.npy`

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.decomposition import TruncatedSVD

DATA_DIR = Path("data/ml-32m")
OUT_DIR = Path("data/processed_32m_ubcf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_PATH = DATA_DIR / "ratings.csv"
MOVIES_PATH = DATA_DIR / "movies.csv"
TAGS_PATH = DATA_DIR / "tags.csv"

CHUNK_SIZE = 2_000_000
N_COMPONENTS = 32
SEED = 42
TOP_TAGS = 120

print(RATINGS_PATH, MOVIES_PATH, TAGS_PATH)

data\ml-32m\ratings.csv data\ml-32m\movies.csv data\ml-32m\tags.csv


In [2]:
# Chunked pass for ID discovery
all_users, all_movies = set(), set()
n_rows = 0

for chunk in pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"], chunksize=CHUNK_SIZE):
    n_rows += len(chunk)
    all_users.update(chunk["userId"].unique().tolist())
    all_movies.update(chunk["movieId"].unique().tolist())

unique_users = np.array(sorted(all_users), dtype=np.int64)
unique_movies = np.array(sorted(all_movies), dtype=np.int64)
user_to_idx = {u: i for i, u in enumerate(unique_users)}
movie_to_idx = {m: i for i, m in enumerate(unique_movies)}

print(f"rows={n_rows:,} users={len(unique_users):,} movies={len(unique_movies):,}")

rows=32,000,204 users=200,948 movies=84,432


In [3]:
# Build user-focused CSR matrix: rows=users, cols=movies
row_parts, col_parts, val_parts = [], [], []
for chunk in pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"], chunksize=CHUNK_SIZE):
    row_parts.append(chunk["userId"].map(user_to_idx).to_numpy(dtype=np.int32, copy=False))
    col_parts.append(chunk["movieId"].map(movie_to_idx).to_numpy(dtype=np.int32, copy=False))
    val_parts.append(chunk["rating"].to_numpy(dtype=np.float32, copy=False))

rows = np.concatenate(row_parts)
cols = np.concatenate(col_parts)
vals = np.concatenate(val_parts)
user_sparse = csr_matrix((vals, (rows, cols)), shape=(len(unique_users), len(unique_movies)), dtype=np.float32)

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)
user_latent = svd.fit_transform(user_sparse).astype(np.float32)

print(user_sparse.shape, user_sparse.nnz)
print(user_latent.shape)

(200948, 84432) 32000204
(200948, 32)


In [4]:
# Content features from movies + tags for hybrid cold-start fallback
movies = pd.read_csv(MOVIES_PATH, usecols=["movieId", "genres"])
movies = movies[movies["movieId"].isin(unique_movies)].copy()
movies["genres"] = movies["genres"].fillna("(no genres listed)")
genre_df = movies["genres"].str.get_dummies(sep="|")
genre_df.index = movies["movieId"].map(movie_to_idx)
genre_df = genre_df.sort_index()

tags = pd.read_csv(TAGS_PATH, usecols=["movieId", "tag"])
tags = tags[tags["movieId"].isin(unique_movies)].copy()
tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()
tags = tags[tags["tag"] != ""]

if len(tags) > 0:
    top_tags = tags["tag"].value_counts().head(TOP_TAGS).index
    tags = tags[tags["tag"].isin(top_tags)]
    tags["v"] = 1
    tag_df = tags.pivot_table(index="movieId", columns="tag", values="v", aggfunc="max", fill_value=0)
    tag_df.index = tag_df.index.map(movie_to_idx)
    tag_df = tag_df.sort_index()
else:
    tag_df = pd.DataFrame()

full_idx = np.arange(len(unique_movies), dtype=np.int32)
genre_df = genre_df.reindex(full_idx, fill_value=0)
tag_df = tag_df.reindex(full_idx, fill_value=0) if not tag_df.empty else pd.DataFrame(index=full_idx)
content_features = pd.concat([genre_df, tag_df], axis=1).astype(np.float32).to_numpy(copy=False)

save_npz(OUT_DIR / "user_sparse.npz", user_sparse)
np.save(OUT_DIR / "user_latent.npy", user_latent)
np.save(OUT_DIR / "content_features.npy", content_features)
with open(OUT_DIR / "id_maps.pkl", "wb") as f:
    pickle.dump({"unique_users": unique_users, "unique_movies": unique_movies, "user_to_idx": user_to_idx, "movie_to_idx": movie_to_idx}, f)

print("Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl")

Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl


## D-Wave Local Feature Selection (Simulator)

D-Wave annealing for feature selection in sparse user data (local sim now, real Leap later per supervisor request).

This section formulates a binary QUBO with one variable per latent feature ($z_i \in \{0,1\}$), where selected features maximize a variance-based proxy while enforcing a feature-budget penalty.

Objective (BQM proxy):
- Minimize $-\sum_i w_i z_i + \lambda(\sum_i z_i-k)^2$
- $w_i$: normalized variance contribution of feature $i$
- $k$: target number of selected latent features

In [5]:
# D-Wave Local Feature Selection (Simulator)
from pathlib import Path
import numpy as np

use_dwave_sim = True

try:
    import dimod
except ImportError as exc:
    raise ImportError(
        "dimod is required for local D-Wave-style simulation. Install with: pip install dimod"
    ) from exc

if "OUT_DIR" not in globals():
    OUT_DIR = Path("data/processed_32m_ubcf")
if "SEED" not in globals():
    SEED = 42

if "user_latent" not in globals():
    user_latent = np.load(OUT_DIR / "user_latent.npy")

X_latent = np.asarray(user_latent, dtype=np.float32)
n_users, n_features = X_latent.shape

# Target selected dimensionality (for 32 latent features, default keeps half)
TARGET_FEATURES = min(16, n_features)
TARGET_FEATURES = max(1, TARGET_FEATURES)

# Proxy objective weights: explained-variance approximation from per-feature variance
feature_var = X_latent.var(axis=0).astype(np.float64)
if np.allclose(feature_var.sum(), 0.0):
    feature_scores = np.ones(n_features, dtype=np.float64) / n_features
else:
    feature_scores = feature_var / feature_var.sum()

# Binary QUBO (BQM): minimize -variance_selected + cardinality penalty
k = int(TARGET_FEATURES)
lambda_cardinality = float(np.max(feature_scores) * 2.0 + 1e-9)

linear = {
    i: float(lambda_cardinality * (1 - 2 * k) - feature_scores[i])
    for i in range(n_features)
}
quadratic = {
    (i, j): float(2.0 * lambda_cardinality)
    for i in range(n_features)
    for j in range(i + 1, n_features)
}
offset = float(lambda_cardinality * (k ** 2))

bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset, vartype=dimod.BINARY)

if use_dwave_sim:
    if n_features <= 20:
        sampler = dimod.ExactSolver()
        sample_set = sampler.sample(bqm)
    else:
        sampler = dimod.SimulatedAnnealingSampler()
        sample_set = sampler.sample(bqm, num_reads=50)
else:
    # Upgrade path for real D-Wave Leap later:
    # sampler = LeapHybridCQMSampler(token=os.getenv('DWAVE_API_TOKEN'))
    raise NotImplementedError("Set use_dwave_sim=True until D-Wave Leap token is available.")

best = sample_set.first.sample
selected_mask = np.array([1 if best[i] == 1 else 0 for i in range(n_features)], dtype=np.int8)

# Enforce exact-k fallback for stable downstream dimensions
if selected_mask.sum() != k:
    top_idx = np.argsort(feature_scores)[-k:]
    selected_mask[:] = 0
    selected_mask[top_idx] = 1

user_latent_selected = X_latent[:, selected_mask.astype(bool)]

np.save(OUT_DIR / "user_latent_selected.npy", user_latent_selected.astype(np.float32))
np.save(OUT_DIR / "selected_mask.npy", selected_mask)

retained_ratio = float(feature_var[selected_mask.astype(bool)].sum() / (feature_var.sum() + 1e-12))
print(f"D-Wave local sim complete | n_features={n_features}, selected={int(selected_mask.sum())}")
print(f"Approx retained variance ratio: {retained_ratio:.4f}")
print("Saved: user_latent_selected.npy, selected_mask.npy")

# Small-slice test (200 users) as quick sanity check
slice_n = min(200, n_users)
X_slice = X_latent[:slice_n]
X_slice_selected = X_slice[:, selected_mask.astype(bool)]
print(f"Slice test OK | input={X_slice.shape}, selected={X_slice_selected.shape}")

D-Wave local sim complete | n_features=32, selected=16
Approx retained variance ratio: 0.8269
Saved: user_latent_selected.npy, selected_mask.npy
Slice test OK | input=(200, 32), selected=(200, 16)
